In [1]:
from paths import *
from nano_maker.nanomaker import NanoMaker
from nano_maker.container.configs import (skeleton_weight_filename, naanobot_weight_filename,
                                          skeleton_config, naanobot_config, radial_config)

In [2]:
skeleton_weight_filename = skeleton_weight_filename
skeleton_cfg = skeleton_config
naanobot_weight_filename = naanobot_weight_filename
naanobot_config = naanobot_config
radial_cfg = radial_config

In [3]:
model = NanoMaker(skeleton_weight_filename=skeleton_weight_filename,
                  naanobot_weight_filename=naanobot_weight_filename,
                  skeleton_cfg=skeleton_config,
                  naanobot_config=naanobot_config,
                  radial_cfg=radial_cfg)

In [4]:
drug_i_want_to_deliver = "CC(C)OC(=O)Nc1cc2cc(c(F)c(N)c2cn1)-c1cnc2OCCNc2c1C"

In [5]:
model.ingest_chemical(drug_i_want_to_deliver)

Chemical Ingested: CC(C)OC(=O)Nc1cc2cc(c(F)c(N)c2cn1)-c1cnc2OCCNc2c1C
Drug Likeness Rules Passed: 4 / 4


In [6]:
pocket_data = model.generate_pocket_data(temperature=0.3)
print(len(pocket_data))
print(type(pocket_data))

4
<class 'dict'>


In [7]:
pocket_data

{'SMILES': 'CC(C)OC(=O)Nc1cc2cc(c(F)c(N)c2cn1)-c1cnc2OCCNc2c1C',
 'Temperature': 0.3,
 '3D_skeleton': [[14.24430673770662, 4.9345609346567985, -0.4124961331417609],
  [13.031343529411895, 6.557200007651261, 2.1170455977006437],
  [8.79106588204945, 11.233353702713465, 0.9434529629398836],
  [4.097200493930941, 12.241898469648177, -4.904369357081438],
  [13.14409178952104, -2.4852151102206075, -3.90734523840236],
  [13.120885654002166, 3.606847700392839, 0.3453699460098587],
  [7.4361367638635185, 10.828480544716404, 0.6591537658178073],
  [8.108399731018347, 9.49990151298287, -2.6674009616610714],
  [11.951954702986354, -0.22854159005811633, 3.8206154856699124],
  [1.682192782674072, 11.954925633443807, -3.1193565872337587],
  [12.023239246918685, -1.2109359357217422, -0.9622281397132273],
  [9.248840600251155, -7.572959161566592, 1.4541697864786078],
  [4.903801184468133, 10.473005216454705, -2.9942888902730136],
  [8.592416143325996, 2.63227503189387, -7.241219226370444],
  [10.49131

In [8]:
smiles_str = pocket_data["SMILES"]
skeleton = pocket_data['3D_skeleton']
aa_seq = pocket_data['aa_ids']

In [9]:
# GENERATION QUALITY ASSESSMENTS

In [10]:
import torch
import torch.nn.functional as F
from nano_maker.naanobot import NAAnoBot
from nano_maker.modules.nAAno.smiles_handler import smiles_fingerprint

@torch.no_grad()
def diagnose_generation(model, map4_enc, sph_coordinates, n=50):
    model.eval()
    map4_enc = map4_enc.to(next(model.parameters()).device)

    bioch_context = [model.naano_module.get_nAAnovector("VOID") for _ in range(model.block_size)]
    coord_context = [[model.max_angstroms, 0, 0] for _ in range(model.block_size)]
    aa_chain = ""
    for i, coordinate in enumerate(sph_coordinates[:n]):
        naano_X = model.naano_module.get_nAAno_X(coord_context, bioch_context, coordinate).unsqueeze(0).to(map4_enc.device)
        output, _ = model.forward(naano_X, map4_enc)

        # print raw predicted vector
        print(f"\nPosition {i}:")
        print(f"  raw output: {output[0].detach().cpu().numpy().round(2)}")

        # print distances to all amino acids
        vector = output[0].detach().float()
        distances = {}
        for aa_id, n_v in model.naano_module.nAAno_vectors.items():
            if aa_id == 'VOID':
                continue
            n_v_t = torch.tensor(n_v, dtype=torch.float32)
            distances[aa_id] = F.mse_loss(vector,n_v_t)


        sorted_distances = sorted(distances.items(), key=lambda x: x[1])
        print(f"  top 3: {sorted_distances[:20]}")

        # update context
        aa_id = model.naano_module.approx_id(output[0], temperature=0.1)
        bioch_context = bioch_context[1:]+[model.naano_module.get_nAAnovector(aa_id)]
        coord_context = coord_context[1:]+[coordinate]
        aa_chain += aa_id
        print(aa_chain)


map4_enc = torch.tensor(smiles_fingerprint(drug_i_want_to_deliver), dtype=torch.float32).unsqueeze(0)
nb_cfg = naanobot_config.copy()
sph_coordinates = model._pocket_sph_skeleton()
naanobot_model = NAAnoBot(n_embd=nb_cfg["n_embd"], n_head=nb_cfg["n_head"],
                                           n_layers=nb_cfg["n_layers"],
                                           block_size=nb_cfg["block_size"],
                                           map4_res=nb_cfg["map4_res"],
                                           n_spatial_features=nb_cfg["n_spatial_features"],
                                           max_angstroms=nb_cfg["max_angstroms"],
                                           dropout=nb_cfg["dropout"])
diagnose_generation(model=naanobot_model, map4_enc=map4_enc, sph_coordinates=sph_coordinates)


Position 0:
  raw output: [-0.84  0.72 -0.43  0.5   1.    0.47  0.37 -0.77  1.1  -0.67  1.3   0.16
  0.19  0.33 -0.12]
  top 3: [('Q', tensor(0.4629)), ('N', tensor(0.4834)), ('T', tensor(0.4864)), ('Y', tensor(0.5029)), ('L', tensor(0.5253)), ('K', tensor(0.5350)), ('R', tensor(0.5359)), ('I', tensor(0.5430)), ('V', tensor(0.5437)), ('F', tensor(0.5441)), ('H', tensor(0.5554)), ('A', tensor(0.5557)), ('S', tensor(0.5717)), ('G', tensor(0.5953)), ('P', tensor(0.6097)), ('W', tensor(0.6251)), ('E', tensor(0.6559)), ('M', tensor(0.6726)), ('C', tensor(0.6807)), ('D', tensor(0.6834))]
F

Position 1:
  raw output: [-0.61  0.75 -0.5   0.34  0.61  0.49  0.46 -0.33  0.92 -0.79  1.05  0.5
 -0.24  0.38 -0.12]
  top 3: [('Q', tensor(0.3611)), ('N', tensor(0.3765)), ('T', tensor(0.4007)), ('K', tensor(0.4313)), ('A', tensor(0.4324)), ('R', tensor(0.4362)), ('L', tensor(0.4504)), ('S', tensor(0.4557)), ('G', tensor(0.4609)), ('Y', tensor(0.4817)), ('H', tensor(0.4820)), ('V', tensor(0.4904)), ('P